# Fast Simulated Annealing

In [11]:
import numpy as np
import math
import time
import ast
import random as rd
from numba import njit

In [12]:
def parse_tsplib(file_path):
    """Parses standard EUC_2D coordinate sections from a .tsp file."""
    clean_lines = (line for line in open(file_path) if len(line.split()) == 3 and line.split()[0].isdigit())
    coords = np.loadtxt(clean_lines, usecols=(1, 2))
    return coords

def get_data(file_path):
    tsp_data = []
    with open(file_path, "r") as f:
        next(f)
        for line in f:
            parts = line.split()
            if parts[2] == "EUC_2D":
                name, nb_city, data_type = parts[0], int(parts[1]), parts[2]
                bound = ast.literal_eval(parts[3])
                tsp_data.append((name, nb_city, data_type, bound))
    return tsp_data


In [13]:
tsp_data = get_data("tsp_data.txt")
print(tsp_data)

[('a280', 280, 'EUC_2D', 2579), ('berlin52', 52, 'EUC_2D', 7542), ('bier127', 127, 'EUC_2D', 118282), ('brd14051', 14051, 'EUC_2D', [468942, 469445]), ('ch130', 130, 'EUC_2D', 6110), ('ch150', 150, 'EUC_2D', 6528), ('d198', 198, 'EUC_2D', 15780), ('d493', 493, 'EUC_2D', 35002), ('d657', 657, 'EUC_2D', 48912), ('d1291', 1291, 'EUC_2D', 50801), ('d1655', 1655, 'EUC_2D', 62128), ('d2103', 2103, 'EUC_2D', [79952, 80450]), ('d15112', 15112, 'EUC_2D', [1564590, 1573152]), ('d18512', 18512, 'EUC_2D', [644650, 645488]), ('eil51', 51, 'EUC_2D', 426), ('eil76', 76, 'EUC_2D', 538), ('eil101', 101, 'EUC_2D', 629), ('fl417', 417, 'EUC_2D', 11861), ('fl1400', 1400, 'EUC_2D', 20127), ('fl1577', 1577, 'EUC_2D', [22204, 22249]), ('fl3795', 3795, 'EUC_2D', [28723, 28772]), ('fnl4461', 4461, 'EUC_2D', 182566), ('gil262', 262, 'EUC_2D', 2378), ('kroA100', 100, 'EUC_2D', 21282), ('kroB100', 100, 'EUC_2D', 22141), ('kroC100', 100, 'EUC_2D', 20749), ('kroD100', 100, 'EUC_2D', 21294), ('kroE100', 100, 'EUC_2D

In [14]:
current_tsp = next((r for r in tsp_data if r[0] == "ch130"))
cities = parse_tsplib(f"tsp_data/{current_tsp[0]}.tsp")
print("Current data:", current_tsp[0])
print(cities[:5])
print("Bound:", current_tsp[3])

Current data: ch130
[[334.59092458 161.78093191]
 [397.64466341 262.81653307]
 [503.87418271 172.87411512]
 [444.04794035 384.64918096]
 [311.61371467   2.00916998]]
Bound: 6110


In [15]:
def simulated_annealing(coords, init_temp, cooling_rate, mini_temp):
    def two_opt(circuit, a, b):
        return circuit[:a] + circuit[a: b][::-1] + circuit[b:]

    def dist(u, v):
        x1, y1 = coords[u]
        x2, y2 = coords[v]
        return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

    n = len(coords)
    circuit = list(range(n)) + [0]
    cost = sum(dist(circuit[i], circuit[i+1]) for i in range(n))
    temp = init_temp

    best = circuit, cost

    while temp > mini_temp:
        loop_per_temp = 50
        for _ in range(loop_per_temp):
            a, b = rd.randint(1, n), rd.randint(1, n)
            if a == b:
                continue
            a, b = min(a, b), max(a, b)

            u1, u2 = circuit[a-1], circuit[a]
            v1, v2 = circuit[b-1], circuit[b]

            len_before = dist(u1, u2) + dist(v1, v2)
            len_after = dist(u1, v1) + dist(u2, v2)
            dE = len_after - len_before

            if dE < 0 or rd.random() < math.exp(-dE/temp):
                newcircuit = two_opt(circuit, a, b)
                newcost = cost + dE
            
                if newcost < best[1]:
                    best = newcircuit, newcost

                circuit = newcircuit
                cost = newcost

        temp *= cooling_rate

    return best

In [16]:
start = time.perf_counter()
res_c, bound = simulated_annealing(cities, 10000, 0.9975, 0.01)
end = time.perf_counter()
print(f"Execution time for N={current_tsp[1]}: {(end-start):0.4f}")
print("Result: ", bound)
print("True bound:", current_tsp[3])

Execution time for N=130: 3.5092
Result:  6701.644570362399
True bound: 6110


In [17]:
NB_RUN = 10
result_array = []
INIT_TEMP = 10_000; COOLING = 0.9975; MINI_TEMP = 0.01
start = time.perf_counter()
for _ in range(NB_RUN):
    _, bound = simulated_annealing(cities, INIT_TEMP, COOLING, MINI_TEMP)
    result_array.append(bound)
end = time.perf_counter()

avg_length = np.mean(result_array)
min_length = np.min(result_array)
print(f"Execution time for N={current_tsp[1]}: {(end-start):0.4f}")
print(result_array)
print(f"Average length: {avg_length}")
print(f"Minimum length: {min_length}")

Execution time for N=130: 34.2962
[6433.464341513769, 6294.343706582325, 6769.794769810503, 6537.110682900433, 6618.852550755986, 6353.5829774562635, 6554.469237117063, 6445.4637654764365, 6414.979720201641, 6541.892639098878]
Average length: 6496.3954390913295
Minimum length: 6294.343706582325
